In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window
import os 
import shutil

In [0]:
fpl_chips_validated = spark.read.format("csv")\
    .option("header", True)\
        .option("inferSchema", True)\
            .load("/Workspace/Users/edwinvictor73@gmail.com/FPL-transfer-research-assistant/src/Data/Validated_Data/fpl_chips_validated.csv")
                                          

In [0]:
fpl_chips_processed = (
    spark.read.format('delta')\
        .load("/Volumes/main/fpl/fpl_data/Processed_Data/fpl_chips_validated.csv/")
)

In [0]:
identifying_new_rows = (
    fpl_chips_validated.join(fpl_chips_processed, on = ["Chip_Name", "Start_Gameweek"],
                             how = "left_anti")
)

In [0]:
identifying_new_rows.display()

In [0]:
updated_records = (
    fpl_chips_validated.alias("source")
    .join(
        fpl_chips_processed.alias("target"),
        on=(
            (col("source.Chip_Name") == col("target.Chip_Name")) &
            (col("source.Start_Gameweek") == col("target.Start_Gameweek"))
        ),
        how="inner"
    )
    .filter(
        ~(
            col("source.End_Gameweek").eqNullSafe(
                col("target.End_Gameweek")
            ) &
            col("source.Chip_Type").eqNullSafe(
                col("target.Chip_Type")
            )
        )
    )
)

display(updated_records)

In [0]:
processed_path = "/Volumes/main/fpl/fpl_data/Processed_Data/fpl_chips_validated.csv/"
processed_table = DeltaTable.forPath(spark, processed_path)
(
    processed_table.alias("target")
    .merge(
        fpl_chips_validated.alias("source"),
        """
        target.Chip_Name = source.Chip_Name
        AND target.Start_Gameweek = source.Start_Gameweek
        """
    )
    .whenMatchedUpdate(
        condition="""
        NOT (target.End_Gameweek <=> source.End_Gameweek)
        OR NOT (target.chip_type <=> source.chip_type)
        """,
        set={
            "End_Gameweek": "source.End_Gameweek",
            "chip_type": "source.chip_type"
        }
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
display(
    spark.read.format("delta").load(processed_path)
)